# Milestone 3 - Modelacao Preditiva 24h por POI

Grupo 2

João Costa | 113564

Sílvia Ribeiro | 129428

Vitória Rodrigues | 130557

Este notebook reorganiza a fase de modelacao preditiva para ficar alinhada com o objetivo do Milestone 3:

- prever o numero de visitantes de cada POI individualmente
- com 24 horas de antecedencia
- evitando leakage e sinais de contexto com overlap temporal fraco

Em vez de um modelo global com fontes externas inconsistentes, o notebook passa a privilegiar:

- validacao walk-forward por POI
- comparacao com baselines temporais simples
- um modelo autoregressivo regularizado com exogenas estaveis


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)


## 1. Caminhos e ficheiros de entrada

O notebook procura automaticamente a pasta central `dataset` a partir da diretoria atual de execução. Os ficheiros principais são:

- `df_preprocessed_eda.csv`
- `fact_visitor_arrivals_departures.csv`
- `fact_digital_engagement.csv`
- `fact_payment_statistics.csv`

In [ ]:
START_DIR = Path.cwd().resolve()
print(f"Diretoria atual de execução: {START_DIR}")

DATA_DIR = None
for root in [START_DIR, *START_DIR.parents]:
    candidate = root / "dataset"
    if candidate.exists() and candidate.is_dir():
        DATA_DIR = candidate.resolve()
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Erro: Não foi possível localizar a pasta central 'dataset'. "
        "Garante que ela existe na raiz do projeto."
    )

print(f"Pasta unificada de dados encontrada em: {DATA_DIR}")

MAIN_DATA_PATH = DATA_DIR / 'df_preprocessed_eda.csv'
ARRIVALS_PATH = DATA_DIR / "fact_visitor_arrivals_departures.csv"
DIGITAL_PATH = DATA_DIR / "fact_digital_engagement.csv"
PAYMENTS_PATH = DATA_DIR / "fact_payment_statistics.csv"

if not MAIN_DATA_PATH.exists():
    raise FileNotFoundError(f"Erro: Ficheiro principal não encontrado em {MAIN_DATA_PATH}")

df = pd.read_csv(MAIN_DATA_PATH)
df['movement_date'] = pd.to_datetime(df['movement_date'])
print(f"Dataset principal carregado com sucesso. Dimensões: {df.shape}")
display(df.head())

## 2. Carregamento dos dados e auditoria de consistencia temporal

Antes de definir o modelo, importa perceber se os sinais de contexto sao estaveis o suficiente para previsao com 24 horas de antecedencia.


In [ ]:
main_df = pd.read_csv(MAIN_DATA_PATH)
main_df["movement_date"] = pd.to_datetime(main_df["movement_date"], errors="coerce")
main_df = main_df.sort_values(["geography_key", "movement_date"]).reset_index(drop=True)

arrivals_df = pd.read_csv(ARRIVALS_PATH, usecols=["departure_date", "passenger_count", "flow_type"])
arrivals_df["departure_date"] = pd.to_datetime(arrivals_df["departure_date"], errors="coerce")
arrivals_daily = (
    arrivals_df.loc[arrivals_df["flow_type"].astype(str).str.lower() == "arrival"]
    .groupby("departure_date", as_index=False)["passenger_count"]
    .sum()
    .rename(columns={"departure_date": "movement_date", "passenger_count": "daily_arrivals_passenger_count"})
)

digital_df = pd.read_csv(DIGITAL_PATH, usecols=["engagement_date", "search_volume", "sentiment_score"])
digital_df["engagement_date"] = pd.to_datetime(digital_df["engagement_date"], errors="coerce")
digital_daily = (
    digital_df.groupby("engagement_date", as_index=False)
    .agg(
        search_volume=("search_volume", "sum"),
        sentiment_score=("sentiment_score", "mean"),
    )
    .rename(columns={"engagement_date": "movement_date"})
)

payments_preview = pd.read_csv(PAYMENTS_PATH, nrows=5)

context_audit_df = (
    main_df[["movement_date"]]
    .drop_duplicates()
    .merge(arrivals_daily, on="movement_date", how="left")
    .merge(digital_daily, on="movement_date", how="left")
    .sort_values("movement_date")
)

audit_summary = pd.DataFrame(
    [
        {
            "signal": "arrivals_passenger_count",
            "missing_ratio": context_audit_df["daily_arrivals_passenger_count"].isna().mean(),
            "overlap_dates": int(context_audit_df["daily_arrivals_passenger_count"].notna().sum()),
        },
        {
            "signal": "search_volume",
            "missing_ratio": context_audit_df["search_volume"].isna().mean(),
            "overlap_dates": int(context_audit_df["search_volume"].notna().sum()),
        },
        {
            "signal": "sentiment_score",
            "missing_ratio": context_audit_df["sentiment_score"].isna().mean(),
            "overlap_dates": int(context_audit_df["sentiment_score"].notna().sum()),
        },
    ]
)

print("Shape do dataset principal:", main_df.shape)
print("Periodo coberto:", main_df["movement_date"].min().date(), "a", main_df["movement_date"].max().date())
print("Numero de POIs:", main_df["geography_key"].nunique())
print()
print("Cobertura temporal dos sinais externos:")
display(audit_summary)

print("Preview da tabela de pagamentos (apenas para justificar exclusao):")
display(payments_preview.head())


## 3. Decisao sobre as variaveis

A reorganizacao do notebook segue tres principios:

1. **Manter apenas variaveis compatíveis com previsao 24h por POI**
2. **Evitar sinais com missing elevado ou mudanca de regime entre treino e teste**
3. **Privilegiar um modelo simples e robusto para series muito curtas**

### Variaveis removidas

- `sentiment_score`: fica totalmente em falta no periodo relevante
- `search_volume`: aparece apenas na parte final da serie e cria assimetria entre treino e avaliacao
- `daily_arrivals_passenger_count` e respetivo lag: overlap temporal insuficiente e forte dependencia de imputacao
- tabela de pagamentos: granularidade mensal e geografias nao alinhadas com o problema diario por POI

### Variaveis mantidas

- calendario: `day_of_week`, `is_weekend`, `is_holiday`
- meteorologia estavel: `weather_temperature_2m_mean`, `weather_precipitation_sum`, `weather_temperature_range`, `has_precipitation`
- memoria da serie: `total_incoming_lag_1`, `total_incoming_lag_2`, `total_incoming_roll3_mean`
- irregularidade temporal: `days_since_prev_obs`

O objetivo e comparar um baseline temporal simples com modelos autoregressivos leves e interpretaveis.


In [ ]:
model_df = main_df.copy()

for column in ["is_weekend", "is_holiday", "has_precipitation"]:
    model_df[column] = model_df[column].astype(int)

model_df["poi_label"] = model_df["poi_name"].replace(
    {
        "Caldeiras das Furnas (Aglomerado Urbano)": "Caldeiras das Furnas",
        "Lagoa do Canário": "Lagoa do Canario",
        "Sete Cidades - Ponte entre Lagoas": "Sete Cidades - Ponte",
        "Vista do Rei - Miradouro": "Vista do Rei",
    }
)

for lag in [1, 2]:
    model_df[f"total_incoming_lag_{lag}"] = (
        model_df.groupby("geography_key")["total_incoming"].shift(lag)
    )

model_df["total_incoming_roll3_mean"] = (
    model_df.groupby("geography_key")["total_incoming"]
    .shift(1)
    .rolling(3, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

model_df["days_since_prev_obs"] = (
    model_df.groupby("geography_key")["movement_date"].diff().dt.days
)

feature_cols = [
    "day_of_week",
    "is_weekend",
    "is_holiday",
    "weather_temperature_2m_mean",
    "weather_precipitation_sum",
    "weather_temperature_range",
    "has_precipitation",
    "total_incoming_lag_1",
    "total_incoming_lag_2",
    "total_incoming_roll3_mean",
    "days_since_prev_obs",
]

preview_cols = ["movement_date", "poi_label", "total_incoming"] + feature_cols
display(model_df[preview_cols].head(12))


## 4. Estrategia de validacao

Como o objetivo e prever **cada POI individualmente com 24 horas de antecedencia**, a avaliacao passa a ser feita com:

- **walk-forward por POI**
- treino apenas com observacoes anteriores
- previsao de um unico ponto seguinte em cada passo

Esta estrategia aproveita melhor o pouco historico disponivel do que um unico split fixo.

O benchmark principal passa a ser:

- `Naive_lag1`: previsao = ultimo valor observado

E os modelos candidatos sao:

- `Linear_ARX`
- `Ridge_ARX`

O modelo `Ridge_ARX` e particularmente adequado aqui por reduzir instabilidade em amostras muito pequenas.


In [ ]:
capacity_df = (
    model_df.groupby(["geography_key", "poi_label"], as_index=False)["total_incoming"]
    .max()
    .rename(columns={"total_incoming": "poi_capacity_max"})
)

models = {
    "Linear_ARX": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LinearRegression()),
        ]
    ),
    "Ridge_ARX": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("model", Ridge(alpha=1.0)),
        ]
    ),
}

def clip_non_negative(values):
    return np.clip(np.asarray(values, dtype=float), a_min=0.0, a_max=None)

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "WAPE": np.abs(y_true - y_pred).sum() / np.abs(y_true).sum(),
        "SMAPE": np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-9)),
    }

walkforward_rows = []
min_train_size = 5

for poi_name, poi_df in model_df.groupby("poi_label"):
    poi_df = poi_df.sort_values("movement_date").reset_index(drop=True)

    for step_idx in range(min_train_size, len(poi_df)):
        train_slice = poi_df.iloc[:step_idx].copy()
        test_slice = poi_df.iloc[step_idx : step_idx + 1].copy()

        y_true = float(test_slice["total_incoming"].iloc[0])
        naive_pred = test_slice["total_incoming_lag_1"].iloc[0]
        if pd.isna(naive_pred):
            naive_pred = train_slice["total_incoming"].iloc[-1]

        base_row = {
            "movement_date": test_slice["movement_date"].iloc[0],
            "geography_key": int(test_slice["geography_key"].iloc[0]),
            "poi_label": poi_name,
            "total_incoming": y_true,
            "poi_capacity_max": float(
                capacity_df.loc[capacity_df["geography_key"] == int(test_slice["geography_key"].iloc[0]), "poi_capacity_max"].iloc[0]
            ),
        }

        walkforward_rows.append(
            {
                **base_row,
                "model": "Naive_lag1",
                "prediction": float(max(0.0, naive_pred)),
            }
        )

        X_train = train_slice[feature_cols]
        y_train = train_slice["total_incoming"]
        X_test = test_slice[feature_cols]

        for model_name, model in models.items():
            fitted_model = model.fit(X_train, y_train)
            prediction = float(clip_non_negative(fitted_model.predict(X_test))[0])
            walkforward_rows.append(
                {
                    **base_row,
                    "model": model_name,
                    "prediction": prediction,
                }
            )

prediction_history_df = pd.DataFrame(walkforward_rows).sort_values(
    ["poi_label", "movement_date", "model"]
).reset_index(drop=True)

print("Numero total de previsoes walk-forward:", len(prediction_history_df))
display(prediction_history_df.head(12))


## 5. Avaliacao dos modelos

O `MAPE` deixa de ser a metrica principal porque valores reais muito baixos distorcem fortemente a interpretacao.

As metricas principais passam a ser:

- `MAE`
- `RMSE`
- `WAPE`
- `SMAPE`


In [ ]:
overall_metrics_df = (
    prediction_history_df.groupby("model")
    .apply(
        lambda frame: pd.Series(
            {
                **compute_metrics(frame["total_incoming"], frame["prediction"]),
                "n_forecasts": len(frame),
            }
        ),
        include_groups=False,
    )
    .reset_index()
    .sort_values(["WAPE", "MAE"])
    .reset_index(drop=True)
)

poi_metrics_df = (
    prediction_history_df.groupby(["poi_label", "model"])
    .apply(
        lambda frame: pd.Series(
            {
                **compute_metrics(frame["total_incoming"], frame["prediction"]),
                "n_forecasts": len(frame),
            }
        ),
        include_groups=False,
    )
    .reset_index()
    .sort_values(["poi_label", "WAPE", "MAE"])
    .reset_index(drop=True)
)

print("Metricas globais:")
display(overall_metrics_df)

print("Metricas por POI:")
display(poi_metrics_df)


## 6. Melhor modelo e suporte operacional

O melhor modelo e escolhido segundo `WAPE`, com `MAE` como criterio de desempate. Isto favorece uma leitura mais estavel do erro em contexto de volumes muito diferentes entre POIs.


In [ ]:
best_model_name = overall_metrics_df.iloc[0]["model"]
print(f"Melhor modelo segundo WAPE: {best_model_name}")

best_prediction_df = (
    prediction_history_df.loc[prediction_history_df["model"] == best_model_name]
    .copy()
    .sort_values(["poi_label", "movement_date"])
    .reset_index(drop=True)
)

def crowding_factor_from_rate(rate):
    if rate <= 0.50:
        return "Baixo"
    if rate <= 0.80:
        return "Moderado"
    if rate <= 1.00:
        return "Elevado"
    return "Critico"

def gstc_a8_alert_from_rate(rate):
    if rate > 0.80:
        return "Risco de Sobrelotacao"
    return "Sem alerta"

best_prediction_df["predicted_occupancy_rate"] = (
    best_prediction_df["prediction"] / best_prediction_df["poi_capacity_max"]
)
best_prediction_df["crowding_factor"] = (
    best_prediction_df["predicted_occupancy_rate"].apply(crowding_factor_from_rate)
)
best_prediction_df["gstc_a8_alert"] = (
    best_prediction_df["predicted_occupancy_rate"].apply(gstc_a8_alert_from_rate)
)

display(best_prediction_df.head(12))


## 7. Conclusao metodologica

Esta reorganizacao assume explicitamente que:

- o objetivo e prever **cada POI individualmente**
- a serie e curta e irregular, pelo que modelos demasiado complexos tendem a ser instaveis
- um modelo autoregressivo regularizado e um ponto de partida mais robusto do que sinais externos com overlap fraco

Se o `Ridge_ARX` superar consistentemente o baseline `Naive_lag1`, entao passa a ser o candidato principal para a fase seguinte.


In [ ]:
recommendation_text = (
    "Recomendacao: usar o modelo autoregressivo regularizado por POI como candidato principal, "
    "mantendo Naive_lag1 como baseline obrigatorio e tratando WAPE, MAE e RMSE como metricas principais."
)

print(recommendation_text)
print("\nResumo do melhor modelo:")
display(overall_metrics_df.head(1))


## 8. Exportacao dos resultados

O notebook exporta:

- metricas globais dos modelos
- metricas por POI
- historico de previsoes walk-forward
- previsoes do melhor modelo com indicadores operacionais


In [ ]:
OUTPUT_DIR = DATA_DIR / "export_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

overall_metrics_path = OUTPUT_DIR / "modelacao_24h_overall_metrics.csv"
poi_metrics_path = OUTPUT_DIR / "modelacao_24h_poi_metrics.csv"
predictions_history_path = OUTPUT_DIR / "modelacao_24h_predictions_history.csv"
best_predictions_path = OUTPUT_DIR / "modelacao_24h_best_model_predictions.csv"

overall_metrics_df.to_csv(overall_metrics_path, index=False)
poi_metrics_df.to_csv(poi_metrics_path, index=False)
prediction_history_df.to_csv(predictions_history_path, index=False)
best_prediction_df.to_csv(best_predictions_path, index=False)

print("Exportação concluída com sucesso para a pasta:")
print(OUTPUT_DIR)
